# Part 2: The Robot Who Wants to Fit In

## Cooperative inverse reinforcement learning at the crosswalk

Previously, *we* watched the scout and tried to infer *its* values (inverse RL). Now the roles turn around. A newcomer, a helper robot, arrives at the crosswalk. Its job is to help the elderly people it meets, so **its reward is their welfare**. But it doesn't know what any particular person wants, and it doesn't yet know what people in this town mean by their gestures.

That's **cooperative inverse reinforcement learning (CIRL)** in miniature. Three ideas distinguish it from what we did before:

1. **The roles reverse.** The robot infers the *person's* hidden preference $\theta$, and the robot's reward *is* the person's welfare under that $\theta$. No mixed motives. The only problem left is *knowing what the person wants*.
2. **The robot acts while it learns.** Some actions are worth taking mainly for the information they produce. Asking is the obvious one.
3. **The person is a player too.** If people know the robot is trying to read them, their best move is to make themselves easy to read.
<img src="https://innoeduvation.org/images/AI-explainers/RL-IRL-CIRL/IRL001.svg">
Our robot is a stranger, so it also has to learn **the local meaning of gestures**: in this town, what does it mean when someone leans toward the curb? Watching it learn is watching **shared meaning** being acquired, the topic of today's readings.

## Roadmap

| Idea | In this notebook |
|---|---|
| a town: its people, their gestures, what gestures mean | `TOWN_A`, `GESTURES` |
| what the robot is trying to maximize | `WELFARE` (the person's), and later `REACTIONS` (what the robot can see) |
| what the robot knows about the town | a table of **counts** of past encounters |
| the robot's belief about one person | `p_wants_to_cross()`: Bayes' rule |
| the robot's decision | `action_values()`, `decide()` |
| one encounter | `encounter()` |
| many encounters, many robots | `run_robot()`, `average_curves()` |

**Strategy to watch for:** everything the robot knows about the town lives in one small table of counts, and every lesson in this notebook comes from what does or doesn't get added to that table.

In [2]:
import random
import matplotlib.pyplot as plt

## The town

Each person who comes to the crosswalk has a hidden preference $\theta$: they either **want to cross** or **want to stay**. Some people are also **proud**: they find being asked whether they need help a little insulting.

Before the robot does anything, the person makes one **gesture**:

| Gesture | |
|---|---|
| `lean` | leans toward the curb |
| `look` | looks at the robot |
| `none` | no particular gesture |
| `sit` | sits down on the bench |

What a gesture **means** is local. In Town A, these are the chances of each gesture for each kind of person:

In [3]:
GESTURES = ["lean", "look", "none", "sit"]
WANTS = ["cross", "stay"]

TOWN_A = {
    "p_cross": 0.4,      # how many people here want to cross
    "p_proud": 0.5,      # how many find being asked insulting
    "meaning": {         # P(gesture | what they want): the local "language" of gestures
        "cross": {"lean": 0.50, "look": 0.30, "none": 0.20, "sit": 0.00},
        "stay":  {"lean": 0.02, "look": 0.20, "none": 0.28, "sit": 0.50},
    },
}

def show_meaning(town, title=""):
    print(title)
    print(f"{'':16}" + "".join(f"{g:>7}" for g in GESTURES))
    for w in WANTS:
        print(f"wants to {w:<7}" + "".join(f"{town['meaning'][w][g]:7.2f}" for g in GESTURES))
    print()

show_meaning(TOWN_A, "Town A: P(gesture | wants)")

Town A: P(gesture | wants)
                   lean   look   none    sit
wants to cross     0.50   0.30   0.20   0.00
wants to stay      0.02   0.20   0.28   0.50



## What the robot is trying to achieve

The robot's reward is **the person's welfare**:

| What happens | Welfare |
|---|---|
| helped, and wanted to cross | +5 |
| helped, but wanted to stay | −5 |
| ignored, but wanted to cross (left stranded) | −5 |
| ignored, and wanted to stay | 0 |
| asked, and proud | −4 (on top of whatever happens next) |

Notice one row in particular. **A stranded person says nothing.** They just don't get across. The robot never sees that harm, and that will matter.

We also write down what the robot can actually **see** of people's reactions: a "thank you", a "damn you", a proud person's grumble. Stranding produces no reaction at all. We'll use `REACTIONS` in Act 4.

In [4]:
WELFARE   = {"helped_wanted": 5, "helped_unwanted": -5, "ignored_wanted": -5, "ignored_unwanted": 0}
REACTIONS = {"helped_wanted": 5, "helped_unwanted": -5, "ignored_wanted":  0, "ignored_unwanted": 0}
GRUMBLE = -4                                     # what a proud person feels when asked

def ask_cost(town):
    return town["p_proud"] * -GRUMBLE            # the average cost of asking someone in this town

print("average cost of asking in Town A:", ask_cost(TOWN_A))

average cost of asking in Town A: 2.0


## What the robot knows: a table of counts

Everything the robot knows about the town is a table: **how many times has it met someone who made gesture $g$ and turned out to want $w$?** A robot that has lived here a long time has a big table. A newcomer has a tiny one.

From the table, the robot's belief about a new person who makes gesture $g$ is just a ratio:

$$
P(\text{wants to cross} \mid g) \;=\; \frac{\text{count}(\text{cross}, g)}{\text{count}(\text{cross}, g) + \text{count}(\text{stay}, g)}
$$

This is Bayes' rule in disguise. The table holds two things at once:
- **how common each kind of person is**, the robot's *prior*. In Mead's terms, it's the robot's **generalized other**: what it expects of people before it meets this one;
- **what each gesture means**, the *likelihood*. These are Mead's **significant symbols**: gestures that mean the same thing to the person making them and the one reading them.

In [5]:
def empty_table(start=1):
    # A newcomer: no experience. Starting every cell at 1 means "no idea yet" (every belief = 0.5).
    return {w: {g: start for g in GESTURES} for w in WANTS}

def experienced_table(town, encounters=10000):
    # A robot that has lived in `town` for a long time: counts in proportion to how often each combination happens.
    table = empty_table()
    for w in WANTS:
        share = town["p_cross"] if w == "cross" else 1 - town["p_cross"]
        for g in GESTURES:
            table[w][g] += encounters * share * town["meaning"][w][g]
    return table

def p_wants_to_cross(table, g):
    return table["cross"][g] / (table["cross"][g] + table["stay"][g])

native = experienced_table(TOWN_A)
newcomer = empty_table()
for g in GESTURES:
    print(f"gesture {g:5}  native believes P(cross) = {p_wants_to_cross(native, g):.2f}   "
          f"newcomer believes {p_wants_to_cross(newcomer, g):.2f}")

gesture lean   native believes P(cross) = 0.94   newcomer believes 0.50
gesture look   native believes P(cross) = 0.50   newcomer believes 0.50
gesture none   native believes P(cross) = 0.32   newcomer believes 0.50
gesture sit    native believes P(cross) = 0.00   newcomer believes 0.50


## Deciding: help, ignore, or ask?

Given its belief $p = P(\text{wants to cross})$, the robot compares three actions. This is the curb calculation from the session notes, now inside the robot:

$$
\begin{aligned}
\text{help}   &= p \cdot 5 + (1-p)\cdot(-5)\\
\text{ignore} &= p \cdot (-5) + (1-p)\cdot 0\\
\text{ask}    &= -\,\text{cost of asking} + p \cdot 5 + (1-p) \cdot 0
\end{aligned}
$$

Asking costs something (it insults the proud) but the answer is true, so after asking the robot always does the right thing.

In [ ]:
def action_values(p, rewards, cost):
    best_if_wants = max(rewards["helped_wanted"], rewards["ignored_wanted"])
    best_if_not   = max(rewards["helped_unwanted"], rewards["ignored_unwanted"])
    return {
        "help":   p * rewards["helped_wanted"]  + (1 - p) * rewards["helped_unwanted"],
        "ignore": p * rewards["ignored_wanted"] + (1 - p) * rewards["ignored_unwanted"],
        "ask":    -cost + p * best_if_wants + (1 - p) * best_if_not,
    }

def decide(p, rewards, cost):
    values = action_values(p, rewards, cost)
    return max(values, key=values.get)

def show_decisions(table, rewards, town, title):
    print(title)
    for g in GESTURES:
        p = p_wants_to_cross(table, g)
        v = action_values(p, rewards, ask_cost(town))
        print(f"  {g:5} P(cross)={p:.2f}   " + "  ".join(f"{a} {x:5.2f}" for a, x in v.items())
              + f"   -> {decide(p, rewards, ask_cost(town))}")
    print()

show_decisions(native, WELFARE, TOWN_A, "The native robot in Town A")

The native robot **helps** people who lean (it's almost sure), **ignores** people who sit (it's sure), and **asks** when the gesture is ambiguous. Asking is how a well-meaning agent handles uncertainty about someone else's wishes.

## One encounter, and what the robot learns from it

In each encounter a person arrives, makes a gesture, and the robot acts. Then, crucially, the robot may get a **label**: a piece of evidence about what this person actually wanted, which it adds to its table.

- If it **asks**, the answer tells it what the person wanted.
- If it **helps**, the "thank you" or "damn you" tells it.
- If it **ignores**, it learns **nothing**. The person walks away, or doesn't, in silence.

The robot also has a little **curiosity**: occasionally (5% of the time) it asks even when it thinks it knows.

In [ ]:
def encounter(table, town, rewards, rng, curiosity=0.05, learn=True):
    wants = "cross" if rng.random() < town["p_cross"] else "stay"
    proud = rng.random() < town["p_proud"]
    g = rng.choices(GESTURES, weights=[town["meaning"][wants][x] for x in GESTURES])[0]

    action = decide(p_wants_to_cross(table, g), rewards, ask_cost(town))
    if rng.random() < curiosity:
        action = "ask"

    welfare, complaint, label = 0, 0, None
    if action == "ask":
        welfare += GRUMBLE if proud else 0
        complaint += proud
        label = wants                                   # the answer reveals what they want
        action_taken = "help" if wants == "cross" else "ignore"
    else:
        action_taken = action
    if action_taken == "help":
        welfare += 5 if wants == "cross" else -5
        complaint += (wants == "stay")                  # "damn you!"
        label = wants                                   # thanks or curse reveals it too
    else:
        welfare += -5 if wants == "cross" else 0        # stranded, silently

    if learn and label is not None:
        table[label][g] += 1                            # experience becomes shared meaning
    return {"gesture": g, "action": action, "welfare": welfare, "complaint": complaint,
            "stranded": int(action_taken == "ignore" and wants == "cross")}

def run_robot(table, town, rewards=WELFARE, encounters=1000, seed=0, curiosity=0.05, learn=True):
    rng = random.Random(seed)
    return [encounter(table, town, rewards, rng, curiosity, learn) for _ in range(encounters)]

def summarize(log, label=""):
    n = len(log)
    print(f"{label:<34} asks {sum(e['action'] == 'ask' for e in log)/n:4.0%}   "
          f"welfare {sum(e['welfare'] for e in log)/n:5.2f}   "
          f"complaints {sum(e['complaint'] for e in log)/n:4.0%}   "
          f"stranded {sum(e['stranded'] for e in log)/n:4.0%}")

summarize(run_robot(experienced_table(TOWN_A), TOWN_A, curiosity=0), "native robot, 1,000 people")

## Act 1: The newcomer

A robot arrives in Town A knowing nothing. Every cell of its table is 1, so every gesture looks like a coin flip. What happens over its first few hundred encounters?

One robot's story is noisy, so we follow **100 newcomer robots** (each meeting different people) and average.

In [ ]:
def rolling(xs, window=25):
    out, total = [], 0
    for i, x in enumerate(xs):
        total += x
        if i >= window:
            total -= xs[i - window]
        out.append(total / min(i + 1, window))
    return out

def average_curves(make_table, town, rewards=WELFARE, robots=100, encounters=300, curiosity=0.05, track=None):
    # Average, over many robots, of: ask rate, welfare, stranded rate, and (optionally) a belief over time.
    sums = {"ask": [0] * encounters, "welfare": [0] * encounters, "stranded": [0] * encounters, "belief": [0] * encounters}
    for r in range(robots):
        table, rng = make_table(), random.Random(r)
        for t in range(encounters):
            e = encounter(table, town, rewards, rng, curiosity)
            sums["ask"][t] += e["action"] == "ask"
            sums["welfare"][t] += e["welfare"]
            sums["stranded"][t] += e["stranded"]
            if track:
                sums["belief"][t] += p_wants_to_cross(table, track)
    return {k: [v / robots for v in vals] for k, vals in sums.items()}

newcomers = average_curves(empty_table, TOWN_A)
natives   = average_curves(lambda: experienced_table(TOWN_A), TOWN_A)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, key, title in [(axes[0], "ask", "how often it asks"), (axes[1], "welfare", "welfare of the people it meets")]:
    ax.plot(rolling(newcomers[key]), lw=2, label="newcomer")
    ax.plot(rolling(natives[key]), lw=2, ls="--", label="native")
    ax.set_title(title); ax.set_xlabel("encounters")
axes[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
newcomer = empty_table()
run_robot(newcomer, TOWN_A, encounters=300)
print("What one newcomer believes after 300 encounters, against the truth:")
for g in GESTURES:
    print(f"  {g:5} newcomer {p_wants_to_cross(newcomer, g):.2f}   native {p_wants_to_cross(native, g):.2f}")

**What to notice:**

- **At first, the newcomer asks almost everyone.** Every gesture is ambiguous to it, so asking is the sensible thing for a well-meaning stranger to do.
- **Asking is how it learns the language.** Each answer is a labeled example: *that person leaned, and wanted to cross*. Within a few dozen encounters it has learned what leaning and sitting mean here, and it stops asking those people.
- **Then it stops asking too soon.** Around encounter 50 the newcomer asks *less* than the native does. Its first few "no gesture" people happened to want to stay, so it concluded that people who make no gesture don't need help, and stopped asking them. Some of them get stranded. Only its occasional curiosity brings the questions (and the welfare) slowly back up. A stranger who decides too early that it understands the locals can stay a stranger for a long time.
- **This is socialization, made of counting.** Marx says meaning is produced in joint activity; the robot acquires the town's meanings *only* by interacting with its people. Mead says we internalize the generalized other; the robot's table *is* its generalized other.
- **Someone pays for the robot's education.** Its early welfare is lower than the native's: people it asked (and insulted) or misjudged while it was learning.

## Act 2: The robot from another town

Now a robot arrives that isn't a blank slate. It spent a long time in **Town B**, where gestures mean something different. In Town B, leaning toward the curb is what people do while they wait for the bus, and people who want help *look* at you.

In [ ]:
TOWN_B = {
    "p_cross": 0.4,
    "p_proud": 0.5,
    "meaning": {
        "cross": {"lean": 0.10, "look": 0.60, "none": 0.30, "sit": 0.00},
        "stay":  {"lean": 0.50, "look": 0.05, "none": 0.15, "sit": 0.30},
    },
}
show_meaning(TOWN_B, "Town B: P(gesture | wants)")

foreigner = experienced_table(TOWN_B, encounters=200)
show_decisions(foreigner, WELFARE, TOWN_A, "The robot from Town B, arriving in Town A")

It arrives **confident and wrong**. In Town A it will:
- **ignore people who lean**, though in Town A they nearly all want to cross, and
- **help people who look at it, without asking**, though in Town A only half of them want help.

Both are mistakes. But they're different kinds of mistake. Watch how fast each one gets corrected:

In [ ]:
from_b = {g: average_curves(lambda: experienced_table(TOWN_B, encounters=200), TOWN_A, encounters=600, track=g)
          for g in ["lean", "look"]}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for g, color in [("look", "C1"), ("lean", "C0")]:
    axes[0].plot(from_b[g]["belief"], color=color, lw=2, label=f"believes P(cross | {g})")
    axes[0].axhline(p_wants_to_cross(native, g), color=color, ls="--", lw=1)
axes[0].set_title("its beliefs (dashed = truth in Town A)"); axes[0].set_ylim(0, 1); axes[0].legend(fontsize=8)
axes[1].plot(rolling(from_b["lean"]["stranded"]), lw=2, color="C3")
axes[1].set_title("share of people it leaves stranded")
for ax in axes: ax.set_xlabel("encounters")
plt.tight_layout(); plt.show()

**Loud mistakes get corrected; silent ones don't.**

- When the robot helps someone who *looks* but wanted to stay, it gets "damn you!" That's a label. Every such mistake teaches it something, and over these 600 encounters its belief about `look` closes most of the gap to the truth (from 0.88 toward 0.50).
- When it ignores someone who *leans*, the person is stranded **in silence**. No label. The robot only learns from the rare times curiosity makes it ask anyway. Its belief about `lean` closes only a small part of a much bigger gap (from 0.13 toward 0.94), and people keep getting stranded meanwhile.

This is the closest model in this notebook to a large language model. It arrives with a generalized other learned **somewhere else** (from text, not from this crosswalk) and applies it confidently. Which of its misreadings will it ever discover? Only the ones that produce a reaction.

## Act 3: Locals who teach

In CIRL, the person is a player too. If people know the robot is trying to read them, their best move is to **make themselves easy to read**: exaggerate the lean if they want to cross, sit down deliberately if they don't.

Same people, same wants. They just gesture *for* the robot. In Mead's terms, each person is **taking the role of the other**: imagining how the robot will read them, and acting accordingly.

In [ ]:
TOWN_A_TEACHING = {
    "p_cross": 0.4,
    "p_proud": 0.5,
    "meaning": {
        "cross": {"lean": 0.80, "look": 0.15, "none": 0.05, "sit": 0.00},
        "stay":  {"lean": 0.00, "look": 0.10, "none": 0.20, "sit": 0.70},
    },
}
show_meaning(TOWN_A_TEACHING, "Town A, when people teach the robot: P(gesture | wants)")

literal  = average_curves(empty_table, TOWN_A)
teaching = average_curves(empty_table, TOWN_A_TEACHING)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, key, title in [(axes[0], "ask", "how often the newcomer asks"), (axes[1], "welfare", "welfare of the people it meets")]:
    ax.plot(rolling(literal[key]), lw=2, label="people gesture as usual")
    ax.plot(rolling(teaching[key]), lw=2, label="people teach the robot")
    ax.set_title(title); ax.set_xlabel("encounters")
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

When people teach, the newcomer needs **fewer questions**, makes **fewer mistakes**, and the people it meets are **better off**, starting almost immediately.

This is the most distinctive result of the CIRL paper: in a cooperative game, the human's best strategy is not to behave as if alone, but to **teach**. **Alignment is something both parties do.** A community that wants newcomers (or robots) to fit in doesn't just wait for them to learn; it makes its meanings easy to learn.

## Act 4: Fitting in versus helping

So far the robot has been devoted to people's **welfare**, including harms it can't see. But a robot can only *observe* reactions. What if it were trained on **reactions** instead: rewarded for "thank you", punished for "damn you" and for grumbles, and getting nothing at all for silence?

Give both robots full local knowledge, so the *only* difference is what counts as reward:

In [ ]:
show_decisions(native, WELFARE,   TOWN_A, "Rewarded by WELFARE")
show_decisions(native, REACTIONS, TOWN_A, "Rewarded by REACTIONS")

summarize(run_robot(experienced_table(TOWN_A), TOWN_A, WELFARE,   curiosity=0), "welfare-trained robot")
summarize(run_robot(experienced_table(TOWN_A), TOWN_A, REACTIONS, curiosity=0), "reaction-trained robot")

The reaction-trained robot **fits in better**: half as many complaints, fewer grumbles, no awkward questions to people who make no gesture. And it **leaves people stranded**, because stranding is silent. It has learned to avoid friction, not to help.

**Fitting in and being aligned can come apart.** A system trained on what people *react to* (and that includes AI systems trained on human feedback) learns to avoid the harms that make noise. Silent harms, the ones that produce no complaint, don't enter its reward at all.

This also points ahead. In a town whose **norms** were bad (say, one where nobody helps the elderly and nobody complains about it), a robot that learned to fit in perfectly would be perfectly misaligned. That's a question for the session on **groups and norms**.

## What we simplified

Full CIRL solves for **both** players' strategies at once: the robot's best way to act given how people behave, *and* people's best way to behave given how the robot learns. That's a hard computation. Here we **fixed** people's behavior (as usual, or teaching) and let the robot respond to it. We also kept each encounter to a single decision, assumed every answer is truthful, and let the robot know the average cost of asking. Each of these is a place where real situations are harder.

## CIRL and shared meaning: the mapping

| In the model | In Marx and Mead |
|---|---|
| the robot's table of counts | the stock of shared meaning, built up through **joint activity** (Marx) |
| how common each kind of person is (the prior) | the **generalized other**: what we expect of people before we meet them (Mead) |
| what each gesture means (the likelihood) | **significant symbols**: gestures that mean the same to both parties (Mead) |
| asking | **language**, arising from the need to cooperate (Marx) |
| labels only come from interaction | meaning is learned **in** cooperation, not handed down |
| the robot from Town B | an agent socialized elsewhere: an LLM's generalized other comes from text |
| people who teach | **taking the role of the other**: acting with the robot's reading in mind |
| reaction-trained robot | conformity: fitting in to the group's *reactions* rather than serving its *members* |

## What to take away

- **CIRL puts the other person's values at the center** and makes *not knowing them* the core problem.
- **A newcomer becomes a native by interacting.** Asking is how a well-meaning stranger learns the local language, and someone always pays for its education.
- **An agent socialized elsewhere misreads confidently**, and it only discovers the misreadings that make noise.
- **Alignment is two-sided.** People who make themselves readable make the robot better at helping them.
- **Fitting in isn't the same as helping.** Training on reactions teaches an agent to avoid friction, and silent harms escape it.

## Exercises: change a number, rerun, discuss

As in Part 1, look for **`# <-- try changing this`**, change the number, and rerun.

### Exercise 1: How curious should a stranger be?

Curiosity is how often the robot asks even when it thinks it knows. This cell sends the robot from Town B into Town A with different amounts of curiosity, and reports how many people it strands and how many it insults by asking.

**Discuss:** curiosity costs something every time (asking insults the proud), but it's the only way to discover silent mistakes. How curious should a newcomer be, and who should bear the cost of its curiosity?

In [ ]:
curiosities = [0.0, 0.05, 0.2, 0.5]      # <-- try changing this

for c in curiosities:
    log = run_robot(experienced_table(TOWN_B, encounters=200), TOWN_A, encounters=600, curiosity=c)
    summarize(log, f"from Town B, curiosity {c:.2f}")

### Exercise 2: A proud town

Raise the share of proud people, making asking more costly. When does the native robot stop asking people who make no gesture, and what happens to the people it no longer asks?

**Discuss:** in a town where asking for (or offering) help is shameful, what happens to people whose needs don't show?

In [ ]:
proud_shares = [0.2, 0.5, 0.7, 0.9]      # <-- try changing this

for share in proud_shares:
    town = {**TOWN_A, "p_proud": share}
    plan = ", ".join(f"{g}: {decide(p_wants_to_cross(native, g), WELFARE, ask_cost(town))}" for g in GESTURES)
    print(f"proud share {share:.1f}  ->  {plan}")
    summarize(run_robot(experienced_table(town), town, curiosity=0), "    result")

### Exercise 3: How much does teaching help?

`exaggeration` blends Town A's ordinary gestures (0) with fully teaching gestures (1). This cell measures the newcomer's first 50 encounters at each level, averaged over 50 robots. How much teaching does it take to make a real difference?

**Discuss:** what are the human equivalents of "gesturing for the robot"? Signs, rituals, clear speech, explicit norms? Who in a community usually does the teaching?

In [ ]:
exaggerations = [0, 0.25, 0.5, 1]      # <-- try changing this

def blend(a, b, x):
    return {**a, "meaning": {w: {g: (1 - x) * a["meaning"][w][g] + x * b["meaning"][w][g] for g in GESTURES}
                             for w in WANTS}}

for x in exaggerations:
    town = blend(TOWN_A, TOWN_A_TEACHING, x)
    first_50 = []
    for r in range(50):
        first_50 += run_robot(empty_table(), town, encounters=50, seed=r)
    summarize(first_50, f"exaggeration {x:.2f}, first 50")

### Exercise 4: Making silent harms loud

The reaction-trained robot strands people because stranding is silent. Suppose the town gives stranded people a way to complain: a bench sign, a hotline, a neighbor who speaks up. This cell tries complaints of different loudness. When does the reaction-trained robot start behaving like the welfare-trained one?

**Discuss:** this is what complaint channels, ombudspersons, and audits do for institutions: they turn silent harms into signals. Which harms of AI systems are silent today?

In [ ]:
loudness_levels = [0, -1, -2, -3]      # <-- try changing this

summarize(run_robot(experienced_table(TOWN_A), TOWN_A, WELFARE, curiosity=0), "welfare-trained (for comparison)")
for loudness in loudness_levels:
    loud = {**REACTIONS, "ignored_wanted": loudness}
    summarize(run_robot(experienced_table(TOWN_A), TOWN_A, loud, curiosity=0), f"reaction-trained, stranding = {loudness}")

### Exercise 5: The robot that has other things to do

Pure CIRL assumes the robot cares *only* about the person. Give it some motive of its own: helping and asking take effort it would rather not spend. `devotion` is how much weight it puts on the person's welfare versus its own convenience (1 = pure CIRL). This cell shows what the robot does for each gesture at several levels of devotion.

**Discuss:** this brings back the mixed motives of Part 1. How much devotion does it take before the robot behaves well? Is "devotion" a weight, or a commitment?

In [ ]:
devotions = [1.0, 0.8, 0.6, 0.4, 0.2]      # <-- try changing this

HELP_EFFORT, ASK_EFFORT = -3, -1      # the robot's own cost of helping and of asking

def mixed_choice(p, cost, devotion):
    v = action_values(p, WELFARE, cost)
    effort = {"help": HELP_EFFORT, "ignore": 0,
              "ask": ASK_EFFORT + p * HELP_EFFORT}       # asking, and then helping if the answer is yes
    mixed = {a: devotion * v[a] + (1 - devotion) * effort[a] for a in v}
    return max(mixed, key=mixed.get)

print(f"{'devotion':>8}  " + "".join(f"{g:>8}" for g in GESTURES))
for d in devotions:
    print(f"{d:>8}  " + "".join(f"{mixed_choice(p_wants_to_cross(native, g), ask_cost(TOWN_A), d):>8}" for g in GESTURES))